# Rodada 2 — PPML com heterogeneidade municipal

Zeros preservados. Hectares e intensidade com offset são estimandos distintos. FE municipal/ano harmonizados; não há interpretação causal do estoque. Convergência não prova ausência de separação multidimensional.

In [1]:
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d
print(len(d),"pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.")

3384 pares na grade; ver ESTRATEGIA.md para escolhas anteriores à estimação.


## 1. Amostra comum e referência linear

In [2]:
s=d.dropna(subset=['pp_agricultura','pp_pastagem','estoque','expo_soja','expo_milho']).copy()
g=s.groupby('id').agg(n=('ano','size'),ag=('ha_agricultura','sum'),pa=('ha_pastagem','sum'))
validos=g.index[(g.n>1)&(g.ag>0)&(g.pa>0)]
excluidos=g.loc[~g.index.isin(validos)].copy();excluidos.to_csv(e.p/'auditoria_exclusoes_naolinear.csv')
s=s[s.id.isin(validos)].copy()
assert s.estoque.gt(0).all()
for u in ['agricultura','pastagem']:assert s.groupby('ano')['ha_'+u].sum().gt(0).all()
print(len(s),'pares;',s.id.nunique(),'municípios; exclusões explícitas:',len(excluidos))

2769 pares; 134 municípios; exclusões explícitas: 3


In [3]:
for uso in ['agricultura','pastagem']:
    r=e.linear('G00_'+uso,'pp_'+uso,['expo_soja','expo_milho'],s)
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','ic95_inf','ic95_sup','N']])

,id,termo,coef,ic95_inf,ic95_sup,N
0,G00_agricultura,expo_soja,0.028791,-0.004711,0.062294,2769
1,G00_agricultura,expo_milho,0.003751,-0.006521,0.014023,2769


,id,termo,coef,ic95_inf,ic95_sup,N
2,G00_pastagem,expo_soja,0.103806,0.038567,0.169044,2769
3,G00_pastagem,expo_milho,0.058373,0.002940,0.113806,2769


## 2. Hectares — PPML sem offset

In [4]:
for uso in ['agricultura','pastagem']:
    e.glm('Q01_'+uso,'ha_'+uso,['expo_soja','expo_milho'],s,tipo='PPML')
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','se','ic95_inf','ic95_sup','p','N']])

,id,termo,coef,se,ic95_inf,ic95_sup,p,N
4,Q01_agricultura,expo_soja,0.101756,0.058170,-0.018578,0.222090,0.093577,2769
5,Q01_agricultura,expo_milho,0.218058,0.062277,0.089229,0.346887,0.001921,2769


,id,termo,coef,se,ic95_inf,ic95_sup,p,N
6,Q01_pastagem,expo_soja,0.039943,0.028896,-0.019833,0.099719,0.180152,2769
7,Q01_pastagem,expo_milho,0.052790,0.024875,0.001331,0.104248,0.044807,2769


## 3. Hectares condicionados ao estoque — offset

Impõe elasticidade um ao estoque inicial, variável endógena. Diferença em relação a taxas não é efeito identificado do estoque.

In [5]:
for uso in ['agricultura','pastagem']:
    e.glm('Q02_'+uso,'ha_'+uso,['expo_soja','expo_milho'],s,tipo='PPML',offset=True)
    display(pd.DataFrame(e.coef).tail(2)[['id','termo','coef','se','ic95_inf','ic95_sup','p','N']])
e.salvar('ppml')

,id,termo,coef,se,ic95_inf,ic95_sup,p,N
8,Q02_agricultura,expo_soja,0.085263,0.057601,-0.033893,0.204419,0.152375,2769
9,Q02_agricultura,expo_milho,0.217186,0.062083,0.088758,0.345614,0.001936,2769


,id,termo,coef,se,ic95_inf,ic95_sup,p,N
10,Q02_pastagem,expo_soja,0.036814,0.029762,-0.024753,0.098381,0.228586,2769
11,Q02_pastagem,expo_milho,0.049314,0.023778,0.000124,0.098503,0.049470,2769
